# Introduction

This project builds an end-to-end machine learning pipeline to predict taxi trip durations in New York City using historical trip data. It covers data preprocessing, feature engineering, exploratory data analysis (EDA), model training, and evaluation to generate accurate travel time predictions.

## Problem Statement

The objective is to predict the duration of each taxi trip based on features such as pickup and dropoff locations, pickup time, passenger count, and other trip-related information. Accurate travel time estimation helps improve route planning, ride-sharing services, and overall transportation efficiency.

In [1]:
%matplotlib inline
import pandas as pd
from datetime import datetime
import pandas as pd
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.linear_model import LinearRegression, Ridge,BayesianRidge
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import mean_squared_error
from math import radians, cos, sin, asin, sqrt
import seaborn as sns
import matplotlib
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [16, 10]

<h2>1.3 Loading the Data</h2><br>

In [2]:
train = pd.read_csv('../input/new-york-city-taxi-with-osrm/train.csv')
test = pd.read_csv('../input/new-york-city-taxi-with-osrm/test.csv')

FileNotFoundError: File b'../input/new-york-city-taxi-with-osrm/train.csv' does not exist

In [ ]:
pd.set_option('display.float_format', lambda x: '%.3f' % x)
train.head()

<h3>1.4.2 Data Summary</h3>
<p>The <code>describe()</code> function is used to generate a statistical summary of the dataset, providing descriptive metrics such as the mean, standard deviation, minimum, maximum, and quartile values for numerical features.</p>

In [ ]:
pd.set_option('display.float_format', lambda x: '%.3f' % x)
train.describe()

In [ ]:
train.info()

<h2>2. Data Preparation</h2>

<p>Data preparation focused on extracting meaningful features that could improve trip duration predictions. Temporal features such as the pickup hour, day of the week, day of the month, and month were derived from the <code>pickup_datetime</code> field to capture traffic patterns, rush hours, weekends, and seasonal effects.</p>

<p>Additional trip-related features, including <code>passenger_count</code>, <code>vendor_id</code>, and <code>store_and_fwd_flag</code>, were analyzed to determine their influence on travel time. Geographic features played a key role, with pickup and dropoff latitude and longitude used to compute trip distance, travel direction, and spatial clusters representing different neighborhoods.</p>

<h3>2.1 Trip Duration Clean-up</h3><br>

<p>As we noted earlier there are some outliers associated with the `trip_duration` variable, specifically a 980 hour maximum trip duration and a minimum of 1 second trip duration. I've decided to exclude data that lies outside 2 standard deviations from the mean. It might be worthwhile looking into what effect excluding up to 4 standard deviations would have on the end-results.</p>

In [ ]:
m = np.mean(train['trip_duration'])
s = np.std(train['trip_duration'])
train = train[train['trip_duration'] <= m + 2*s]
train = train[train['trip_duration'] >= m - 2*s]

<h3>2.2 Latitude and Longitude Filtering</h3>

<p>To improve data quality, trips with pickup or dropoff coordinates outside the geographical boundaries of New York City were removed. The dataset was filtered using approximate city latitude and longitude limits, eliminating outliers and ensuring that the model was trained only on valid trip locations within NYC.</p>

In [ ]:
train = train[train['pickup_longitude'] <= -73.75]
train = train[train['pickup_longitude'] >= -74.03]
train = train[train['pickup_latitude'] <= 40.85]
train = train[train['pickup_latitude'] >= 40.63]
train = train[train['dropoff_longitude'] <= -73.75]
train = train[train['dropoff_longitude'] >= -74.03]
train = train[train['dropoff_latitude'] <= 40.85]
train = train[train['dropoff_latitude'] >= 40.63]

[](http://)<h3>2.3 Date Clean-up</h3>
<p>As a final step in preparing our data we need to change the formatting of the date variables (`pickup_datetime` and `dropoff_datetime`). This will help a lot with data extraction in the coming section.</p>

In [ ]:
train['pickup_datetime'] = pd.to_datetime(train.pickup_datetime)
test['pickup_datetime'] = pd.to_datetime(test.pickup_datetime)
train.loc[:, 'pickup_date'] = train['pickup_datetime'].dt.date
test.loc[:, 'pickup_date'] = test['pickup_datetime'].dt.date
train['dropoff_datetime'] = pd.to_datetime(train.dropoff_datetime) #Not in Test

<h2>3. Data Visualization and Analysis</h2>

<p>Exploratory Data Analysis (EDA) was performed to better understand the dataset, identify patterns, detect outliers, and evaluate feature distributions. Visualizations provide valuable insights that help guide feature engineering and model development.</p>

<h3>3.1 Initial Analysis</h3>

<p>An initial histogram of the trip duration was plotted to examine its distribution. This visualization helps identify the range of trip durations, detect skewness and outliers, and determine whether transformations are needed before training the machine learning model.</p>

In [ ]:
plt.hist(train['trip_duration'].values, bins=100)
plt.xlabel('trip_duration')
plt.ylabel('number of train records')
plt.show()

In [ ]:
train['log_trip_duration'] = np.log(train['trip_duration'].values + 1)
plt.hist(train['log_trip_duration'].values, bins=100)
plt.xlabel('log(trip_duration)')
plt.ylabel('number of train records')
plt.show()
sns.distplot(train["log_trip_duration"], bins =100)

<h3>3.2 Trip Trends Over Time</h3>

<p>A time series analysis was performed to examine how the number of taxi trips changes over time. This helps identify temporal trends, seasonal patterns, potential missing data, and unusual spikes or drops in trip volume.</p>

<p>The trip counts for both the training and test datasets were visualized to compare their distributions over time. Similar trends across both datasets indicate that the test set is representative of the training data, helping ensure consistent model performance.</p>

In [ ]:
plt.plot(train.groupby('pickup_date').count()[['id']], 'o-', label='train')
plt.plot(test.groupby('pickup_date').count()[['id']], 'o-', label='test')
plt.title('Trips over Time.')
plt.legend(loc=0)
plt.ylabel('Trips')
plt.show()

<h3>3.3 Dataset Consistency and Vendor Analysis</h3>

<p>The training and test datasets exhibit similar trends over time, indicating that the test set is representative of the training data. A few noticeable drops in trip volume were observed, which may be attributed to external events, data collection issues, or other anomalies. These periods were noted as potential outliers for further investigation.</p>

<p>To better understand feature importance, the average trip duration was also compared across different vendors. This analysis helps determine whether the service provider has a meaningful impact on trip duration and whether <code>vendor_id</code> should be retained as a predictive feature.</p>

In [ ]:
import warnings
warnings.filterwarnings("ignore")
plot_vendor = train.groupby('vendor_id')['trip_duration'].mean()
plt.subplots(1,1,figsize=(17,10))
plt.ylim(ymin=800)
plt.ylim(ymax=840)
sns.barplot(plot_vendor.index,plot_vendor.values)
plt.title('Time per Vendor')
plt.legend(loc=0)
plt.ylabel('Time in Seconds')

<h3>3.4 Vendor and Store-and-Forward Analysis</h3>

<p>The comparison of average trip durations across vendors showed minimal differences, suggesting that <code>vendor_id</code> alone has little influence on travel time. To further evaluate potential operational differences, the <code>store_and_fwd_flag</code> feature was also analyzed to determine whether trips affected by delayed data transmission exhibit different duration patterns.</p>

In [ ]:
snwflag = train.groupby('store_and_fwd_flag')['trip_duration'].mean()

plt.subplots(1,1,figsize=(17,10))
plt.ylim(ymin=0)
plt.ylim(ymax=1100)
plt.title('Time per store_and_fwd_flag')
plt.legend(loc=0)
plt.ylabel('Time in Seconds')
sns.barplot(snwflag.index,snwflag.values)

<h3>3.5 Store-and-Forward and Passenger Count Analysis</h3>

<p>The <code>store_and_fwd_flag</code> showed noticeable differences in average trip duration, indicating that delayed data transmission may be associated with longer trips or operational variations. This feature was therefore considered as a potential predictor in the model.</p>

<p>The relationship between <code>passenger_count</code> and trip duration was also examined by comparing the average travel time across different passenger counts. This analysis helps determine whether the number of passengers has a meaningful impact on trip duration and should be included as a predictive feature.</p>

In [ ]:
pc = train.groupby('passenger_count')['trip_duration'].mean()

plt.subplots(1,1,figsize=(17,10))
plt.ylim(ymin=0)
plt.ylim(ymax=1100)
plt.title('Time per store_and_fwd_flag')
plt.legend(loc=0)
plt.ylabel('Time in Seconds')
sns.barplot(pc.index,pc.values)

<h3>3.6 Passenger Count Distribution</h3>

<p>The analysis showed that <code>passenger_count</code> has little impact on average trip duration, suggesting it is not a strong standalone predictor. A small number of trips recorded with zero passengers were identified as likely data quality issues or anomalies.</p>

<p>To ensure consistency between the datasets, the distribution of passenger counts was compared across the training and test sets. Similar distributions indicate that both datasets are representative and suitable for model training and evaluation.</p>

In [ ]:
train.groupby('passenger_count').size()

In [ ]:
test.groupby('passenger_count').size()

<h3>3.7 Dataset Consistency and Spatial Analysis</h3>

<p>The passenger count distributions in the training and test datasets were largely consistent, indicating that both datasets share similar characteristics. A few uncommon passenger counts, such as trips with nine passengers, were identified as potential outliers but occurred infrequently.</p>

<p>With the categorical and temporal features analyzed, the next step focuses on exploring the spatial characteristics of the data. Pickup and dropoff coordinates are visualized to better understand trip patterns and geographic distributions, laying the foundation for location-based feature engineering.</p>

<h3>3.2 Coordinate Mapping</h3>

<p>The spatial distribution of taxi trips was analyzed by visualizing pickup and dropoff locations. Comparing the geographic patterns in the training and test datasets helps verify that both datasets represent similar trip distributions across New York City.</p>

<h3>3.2.1 Pickup Locations</h3>

<p>Pickup coordinates were plotted on a scatter plot using the predefined geographic boundaries of New York City. This visualization highlights areas with high trip density, identifies potential outliers, and provides insights into the spatial characteristics of the dataset.</p>

In [ ]:
city_long_border = (-74.03, -73.75)
city_lat_border = (40.63, 40.85)
fig, ax = plt.subplots(ncols=2, sharex=True, sharey=True)
ax[0].scatter(train['pickup_longitude'].values[:100000], train['pickup_latitude'].values[:100000],
              color='blue', s=1, label='train', alpha=0.1)
ax[1].scatter(test['pickup_longitude'].values[:100000], test['pickup_latitude'].values[:100000],
              color='green', s=1, label='test', alpha=0.1)
fig.suptitle('Train and test area complete overlap.')
ax[0].legend(loc=0)
ax[0].set_ylabel('latitude')
ax[0].set_xlabel('longitude')
ax[1].set_xlabel('longitude')
ax[1].legend(loc=0)
plt.ylim(city_lat_border)
plt.xlim(city_long_border)
plt.show()

<h3>3.2.2 Distance and Direction Features</h3>

<p>The pickup location distributions in the training and test datasets were highly similar, indicating that both datasets capture comparable spatial patterns across New York City. As expected, the training set contains a larger number of observations.</p>

<p>To enhance the predictive power of the model, additional geographic features were engineered from the pickup and dropoff coordinates. These include the trip distance, travel direction (bearing), and other spatial relationships, which provide valuable information about the characteristics of each taxi journey.</p>

In [ ]:
def haversine_array(lat1, lng1, lat2, lng2):
    lat1, lng1, lat2, lng2 = map(np.radians, (lat1, lng1, lat2, lng2))
    AVG_EARTH_RADIUS = 6371  # in km
    lat = lat2 - lat1
    lng = lng2 - lng1
    d = np.sin(lat * 0.5) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(lng * 0.5) ** 2
    h = 2 * AVG_EARTH_RADIUS * np.arcsin(np.sqrt(d))
    return h

def dummy_manhattan_distance(lat1, lng1, lat2, lng2):
    a = haversine_array(lat1, lng1, lat1, lng2)
    b = haversine_array(lat1, lng1, lat2, lng1)
    return a + b

def bearing_array(lat1, lng1, lat2, lng2):
    AVG_EARTH_RADIUS = 6371  # in km
    lng_delta_rad = np.radians(lng2 - lng1)
    lat1, lng1, lat2, lng2 = map(np.radians, (lat1, lng1, lat2, lng2))
    y = np.sin(lng_delta_rad) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(lng_delta_rad)
    return np.degrees(np.arctan2(y, x))

<h3>3.2.3 Geographic Feature Engineering</h3>

<p>Several location-based features were engineered from the pickup and dropoff coordinates to better capture the characteristics of each trip. These include the Haversine distance (straight-line distance), Manhattan distance (grid-based travel approximation), and trip bearing (direction of travel).</p>

<p>To further represent spatial patterns, geographic clustering was applied to group nearby locations into neighborhoods. These clusters help the model learn location-specific traffic and travel behaviors, improving prediction accuracy.</p>

In [ ]:
train.loc[:, 'distance_haversine'] = haversine_array(train['pickup_latitude'].values, train['pickup_longitude'].values, train['dropoff_latitude'].values, train['dropoff_longitude'].values)
test.loc[:, 'distance_haversine'] = haversine_array(test['pickup_latitude'].values, test['pickup_longitude'].values, test['dropoff_latitude'].values, test['dropoff_longitude'].values)    
    
train.loc[:, 'distance_dummy_manhattan'] =  dummy_manhattan_distance(train['pickup_latitude'].values, train['pickup_longitude'].values, train['dropoff_latitude'].values, train['dropoff_longitude'].values)
test.loc[:, 'distance_dummy_manhattan'] =  dummy_manhattan_distance(test['pickup_latitude'].values, test['pickup_longitude'].values, test['dropoff_latitude'].values, test['dropoff_longitude'].values)

train.loc[:, 'direction'] = bearing_array(train['pickup_latitude'].values, train['pickup_longitude'].values, train['dropoff_latitude'].values, train['dropoff_longitude'].values)
test.loc[:, 'direction'] = bearing_array(test['pickup_latitude'].values, test['pickup_longitude'].values, test['dropoff_latitude'].values, test['dropoff_longitude'].values)

<h3>3.3.3 Creating Location Clusters</h3>

<p>To capture location-based travel patterns, K-Means clustering was applied to the pickup and dropoff coordinates. Instead of relying on predefined neighborhood boundaries, the algorithm groups nearby geographic points into clusters, allowing the model to learn spatial relationships directly from the data.</p>

<p>The clustering process involves three steps: combining the pickup and dropoff coordinates into a single dataset, configuring the MiniBatch K-Means algorithm, and assigning each trip to its corresponding geographic cluster. These cluster labels are then used as additional features for model training.</p>

In [ ]:
coords = np.vstack((train[['pickup_latitude', 'pickup_longitude']].values,
                    train[['dropoff_latitude', 'dropoff_longitude']].values))

In [ ]:
sample_ind = np.random.permutation(len(coords))[:500000]
kmeans = MiniBatchKMeans(n_clusters=100, batch_size=10000).fit(coords[sample_ind])

In [ ]:
train.loc[:, 'pickup_cluster'] = kmeans.predict(train[['pickup_latitude', 'pickup_longitude']])
train.loc[:, 'dropoff_cluster'] = kmeans.predict(train[['dropoff_latitude', 'dropoff_longitude']])
test.loc[:, 'pickup_cluster'] = kmeans.predict(test[['pickup_latitude', 'pickup_longitude']])
test.loc[:, 'dropoff_cluster'] = kmeans.predict(test[['dropoff_latitude', 'dropoff_longitude']])

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1)
ax.scatter(train.pickup_longitude.values[:500000], train.pickup_latitude.values[:500000], s=10, lw=0,
           c=train.pickup_cluster[:500000].values, cmap='autumn', alpha=0.2)
ax.set_xlim(city_long_border)
ax.set_ylim(city_lat_border)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.show()

<h3>3.3.4 Cluster Visualization</h3>

<p>The K-Means clustering results provide a clear spatial representation of New York City by grouping nearby pickup and dropoff locations into geographic clusters. These clusters act as neighborhood-level features, enabling the model to capture location-specific travel patterns and improve prediction accuracy.</p>

<h3>3.4 Date Feature Extraction</h3>

<p>Temporal features were extracted from the <code>pickup_datetime</code> field, including the month, day, weekday, and hour of the trip. These features were then encoded into a machine-learning-friendly format, allowing the model to capture recurring traffic patterns, seasonal effects, and peak travel hours. The distributions of these extracted features were also compared between the training and test datasets to ensure consistency.</p>

In [ ]:
#Extracting Month
train['Month'] = train['pickup_datetime'].dt.month
test['Month'] = test['pickup_datetime'].dt.month

In [ ]:
train.groupby('Month').size(),test.groupby('Month').size()

In [ ]:
train['DayofMonth'] = train['pickup_datetime'].dt.day
test['DayofMonth'] = test['pickup_datetime'].dt.day
len(train.groupby('DayofMonth').size()),len(test.groupby('DayofMonth').size())

In [ ]:
train['Hour'] = train['pickup_datetime'].dt.hour
test['Hour'] = test['pickup_datetime'].dt.hour
len(train.groupby('Hour').size()),len(test.groupby('Hour').size())

In [ ]:
train['dayofweek'] = train['pickup_datetime'].dt.dayofweek
test['dayofweek'] = test['pickup_datetime'].dt.dayofweek
len(train.groupby('dayofweek').size()),len(test.groupby('dayofweek').size())

<h3>3.4.1 Temporal Feature Analysis</h3>

<p>The extracted date features were consistent across both the training and test datasets, making them suitable for feature encoding and model training. These temporal attributes provide valuable information about recurring traffic patterns and seasonal trends.</p>

<p>To better understand their impact, the relationship between average trip speed and time was analyzed across different hours of the day, days of the week, and months of the year. Since average speed is derived directly from trip distance and duration, it serves only as an exploratory metric and is excluded from the final model to prevent data leakage.</p>

In [ ]:
train.loc[:, 'avg_speed_h'] = 1000 * train['distance_haversine'] / train['trip_duration']
train.loc[:, 'avg_speed_m'] = 1000 * train['distance_dummy_manhattan'] / train['trip_duration']
fig, ax = plt.subplots(ncols=3, sharey=True)
ax[0].plot(train.groupby('Hour').mean()['avg_speed_h'], 'bo-', lw=2, alpha=0.7)
ax[1].plot(train.groupby('dayofweek').mean()['avg_speed_h'], 'go-', lw=2, alpha=0.7)
ax[2].plot(train.groupby('Month').mean()['avg_speed_h'], 'ro-', lw=2, alpha=0.7)
ax[0].set_xlabel('Hour of Day')
ax[1].set_xlabel('Day of Week')
ax[2].set_xlabel('Month of Year')
ax[0].set_ylabel('Average Speed')
fig.suptitle('Average Traffic Speed by Date-part')
plt.show()

<h3>3.4.2 Average Speed Analysis</h3>

<p>Average trip speed was analyzed across different time intervals to identify traffic patterns throughout the day, week, and year. The results show lower average speeds during daytime working hours, reflecting increased traffic congestion, while speeds generally increase during evenings and weekends when road traffic is lighter.</p>

<p>Seasonal variations were also observed, with changes in average speed likely influenced by fluctuations in trip volume and road conditions. To further explore these spatial patterns, the next analysis visualizes the relationship between pickup locations and average travel speed across New York City.</p>

In [ ]:
train.loc[:, 'pickup_lat_bin'] = np.round(train['pickup_latitude'], 3)
train.loc[:, 'pickup_long_bin'] = np.round(train['pickup_longitude'], 3)
# Average speed for regions
gby_cols = ['pickup_lat_bin', 'pickup_long_bin']
coord_speed = train.groupby(gby_cols).mean()[['avg_speed_h']].reset_index()
coord_count = train.groupby(gby_cols).count()[['id']].reset_index()
coord_stats = pd.merge(coord_speed, coord_count, on=gby_cols)
coord_stats = coord_stats[coord_stats['id'] > 100]
fig, ax = plt.subplots(ncols=1, nrows=1)
ax.scatter(train.pickup_longitude.values[:500000], train.pickup_latitude.values[:500000], color='black', s=1, alpha=0.5)
ax.scatter(coord_stats.pickup_long_bin.values, coord_stats.pickup_lat_bin.values, c=coord_stats.avg_speed_h.values,
           cmap='RdYlGn', s=20, alpha=0.5, vmin=1, vmax=8)
ax.set_xlim(city_long_border)
ax.set_ylim(city_lat_border)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.title('Average speed')
plt.show()

<h3>3.4.3 Spatial Speed Analysis</h3>

<p>The analysis revealed clear differences in average travel speed across geographic clusters. Trips originating from or passing through central areas of New York City generally experienced lower average speeds due to higher traffic density, while outer regions exhibited faster travel speeds.</p>

<p>These findings suggest that location-based clustering is a valuable feature for predicting trip duration, as it effectively captures spatial variations in traffic conditions and travel behavior.</p>

<h2>4. Data Enrichment and Feature Engineering</h2>

<p>In addition to the engineered temporal and geographic features, external routing information was incorporated to further improve model performance. Feature engineering and data enrichment provide the model with additional contextual information beyond the original dataset.</p>

<h3>4.1 Data Enrichment</h3>

<p>Route-based features from the <strong>Open Source Routing Machine (OSRM)</strong> were integrated into the dataset. These features, such as estimated travel distance, route duration, and the number of route steps, provide valuable information about the optimal driving path and help improve the accuracy of trip duration predictions.</p>

In [ ]:
fr1 = pd.read_csv('../input/new-york-city-taxi-with-osrm/fastest_routes_train_part_1.csv', usecols=['id', 'total_distance', 'total_travel_time',  'number_of_steps'])
fr2 = pd.read_csv('../input/new-york-city-taxi-with-osrm/fastest_routes_train_part_2.csv', usecols=['id', 'total_distance', 'total_travel_time', 'number_of_steps'])
test_street_info = pd.read_csv('../input/new-york-city-taxi-with-osrm/fastest_routes_test.csv',
                               usecols=['id', 'total_distance', 'total_travel_time', 'number_of_steps'])
train_street_info = pd.concat((fr1, fr2))
train = train.merge(train_street_info, how='left', on='id')
test = test.merge(test_street_info, how='left', on='id')

In [ ]:
train.shape, test.shape

<h3>4.2 Data Validation</h3>

<p>Before model training, the processed training and test datasets were compared to ensure they shared a consistent feature set. As expected, the training dataset contains the target variable (<code>trip_duration</code>) and a few additional derived features used only during analysis, while the remaining features are aligned between both datasets.</p>

<h3>4.3 One-Hot Encoding</h3>

<p>Categorical features were converted into numerical representations using one-hot encoding. This process transforms each category into a binary indicator variable, enabling machine learning models such as XGBoost to effectively utilize categorical information during training without introducing ordinal relationships.</p>

In [ ]:
vendor_train = pd.get_dummies(train['vendor_id'], prefix='vi', prefix_sep='_')
vendor_test = pd.get_dummies(test['vendor_id'], prefix='vi', prefix_sep='_')
passenger_count_train = pd.get_dummies(train['passenger_count'], prefix='pc', prefix_sep='_')
passenger_count_test = pd.get_dummies(test['passenger_count'], prefix='pc', prefix_sep='_')
store_and_fwd_flag_train = pd.get_dummies(train['store_and_fwd_flag'], prefix='sf', prefix_sep='_')
store_and_fwd_flag_test = pd.get_dummies(test['store_and_fwd_flag'], prefix='sf', prefix_sep='_')
cluster_pickup_train = pd.get_dummies(train['pickup_cluster'], prefix='p', prefix_sep='_')
cluster_pickup_test = pd.get_dummies(test['pickup_cluster'], prefix='p', prefix_sep='_')
cluster_dropoff_train = pd.get_dummies(train['dropoff_cluster'], prefix='d', prefix_sep='_')
cluster_dropoff_test = pd.get_dummies(test['dropoff_cluster'], prefix='d', prefix_sep='_')

month_train = pd.get_dummies(train['Month'], prefix='m', prefix_sep='_')
month_test = pd.get_dummies(test['Month'], prefix='m', prefix_sep='_')
dom_train = pd.get_dummies(train['DayofMonth'], prefix='dom', prefix_sep='_')
dom_test = pd.get_dummies(test['DayofMonth'], prefix='dom', prefix_sep='_')
hour_train = pd.get_dummies(train['Hour'], prefix='h', prefix_sep='_')
hour_test = pd.get_dummies(test['Hour'], prefix='h', prefix_sep='_')
dow_train = pd.get_dummies(train['dayofweek'], prefix='dow', prefix_sep='_')
dow_test = pd.get_dummies(test['dayofweek'], prefix='dow', prefix_sep='_')

In [ ]:
vendor_train.shape,vendor_test.shape

In [ ]:
passenger_count_train.shape,passenger_count_test.shape

In [ ]:
store_and_fwd_flag_train.shape,store_and_fwd_flag_test.shape

In [ ]:
cluster_pickup_train.shape,cluster_pickup_test.shape

In [ ]:
cluster_dropoff_train.shape,cluster_dropoff_test.shape

In [ ]:
month_train.shape,month_test.shape

In [ ]:
dom_train.shape,dom_test.shape

In [ ]:
hour_train.shape,hour_test.shape

In [ ]:
dow_train.shape,dow_test.shape

<h3>4.4 Feature Alignment</h3>

<p>After one-hot encoding, the training and test datasets were compared to ensure their feature columns matched. A minor inconsistency was observed in the <code>passenger_count</code> feature due to a few rare trips with nine passengers appearing only in the test set.</p>

<p>Since these records were identified as uncommon outliers, they were removed to maintain consistent feature representations between the training and test datasets before model training.</p>

In [ ]:
passenger_count_test = passenger_count_test.drop('pc_9', axis = 1)

<h3>4.5 Final Dataset Preparation</h3>

<p>After completing data cleaning, feature engineering, and encoding, the dataset was prepared for model training. The original categorical features were removed, as they had been replaced by their one-hot encoded representations, and the final training and test datasets were compiled with a consistent set of numerical features.</p>

<p>This finalized dataset serves as the input for the machine learning models, ensuring that all features are properly formatted and ready for training and prediction.</p>

In [ ]:
train = train.drop(['id','vendor_id','passenger_count','store_and_fwd_flag','Month','DayofMonth','Hour','dayofweek',
                   'pickup_longitude','pickup_latitude','dropoff_longitude','dropoff_latitude'],axis = 1)
Test_id = test['id']
test = test.drop(['id','vendor_id','passenger_count','store_and_fwd_flag','Month','DayofMonth','Hour','dayofweek',
                   'pickup_longitude','pickup_latitude','dropoff_longitude','dropoff_latitude'], axis = 1)

train = train.drop(['dropoff_datetime','avg_speed_h','avg_speed_m','pickup_lat_bin','pickup_long_bin','trip_duration'], axis = 1)

In [ ]:
train.shape,test.shape

<p>Now let's add the indicator variables to our datasets.</p>

In [ ]:
Train_Master = pd.concat([train,
                          vendor_train,
                          passenger_count_train,
                          store_and_fwd_flag_train,
                          cluster_pickup_train,
                          cluster_dropoff_train,
                         month_train,
                         dom_train,
                          hour_test,
                          dow_train
                         ], axis=1)

In [ ]:
Test_master = pd.concat([test, 
                         vendor_test,
                         passenger_count_test,
                         store_and_fwd_flag_test,
                         cluster_pickup_test,
                         cluster_dropoff_test,
                         month_test,
                         dom_test,
                          hour_test,
                          dow_test], axis=1)

In [ ]:
Train_Master.shape,Test_master.shape

<p>There're two more columns we can drop because we keep the information from those two columns in other variables: `pickup_datetime` and `pickup_date`.</p>

In [ ]:
Train_Master = Train_Master.drop(['pickup_datetime','pickup_date'],axis = 1)
Test_master = Test_master.drop(['pickup_datetime','pickup_date'],axis = 1)

In [ ]:
Train_Master.shape,Test_master.shape

<h2>5. Model Preparation</h2>

<p>The final dataset was verified before training, ensuring that the training and test sets contained the same feature columns, with the exception of the target variable (<code>trip_duration</code>) present only in the training set.</p>

<p>To evaluate model performance and tune hyperparameters, the training data was split into training and validation subsets using an 80:20 ratio. This validation set allows the model to be evaluated during training, helping prevent overfitting and enabling the selection of optimal model parameters before generating predictions on the unseen test data.</p>

In [ ]:
Train, Test = train_test_split(Train_Master[0:100000], test_size = 0.2)

<h3>5.1 Dataset Finalization</h3>

<p>Before training the model, the datasets were finalized by removing unnecessary features, such as the logarithmic transformation of the target variable, to avoid redundancy. The dataset indices were also reset to ensure consistent row references and simplify subsequent processing during model training and evaluation.</p>

In [ ]:
X_train = Train.drop(['log_trip_duration'], axis=1)
Y_train = Train["log_trip_duration"]
X_test = Test.drop(['log_trip_duration'], axis=1)
Y_test = Test["log_trip_duration"]

Y_test = Y_test.reset_index().drop('index',axis = 1)
Y_train = Y_train.reset_index().drop('index',axis = 1)

<h3>5.2 Preparing Data for XGBoost</h3>

<p>The final preprocessing step involved converting the training, validation, and test datasets into XGBoost's optimized <code>DMatrix</code> format. This data structure improves memory efficiency and training speed while enabling the model to use the training set for learning, the validation set for performance monitoring and hyperparameter tuning, and the test set for generating final trip duration predictions.</p>

In [ ]:
dtrain = xgb.DMatrix(X_train, label=Y_train)
dvalid = xgb.DMatrix(X_test, label=Y_test)
dtest = xgb.DMatrix(Test_master)
watchlist = [(dtrain, 'train'), (dvalid, 'valid')]

<h2>5. Model Training with XGBoost</h2>

<p>With the data fully prepared, the XGBoost regression model was trained to predict taxi trip durations. XGBoost was selected for its ability to efficiently handle large datasets, model complex non-linear relationships, and deliver strong predictive performance on structured tabular data.</p>

<p>The model was trained using the training set and evaluated on the validation set to monitor performance and tune hyperparameters. Parameters such as learning rate, maximum tree depth, subsampling ratio, and the number of boosting rounds were adjusted to minimize the Root Mean Squared Error (RMSE) and improve generalization before generating predictions on the test dataset.</p>

In [ ]:
xgb_pars = {'min_child_weight': 1, 'eta': 0.5, 'colsample_bytree': 0.9, 
            'max_depth': 6,
'subsample': 0.9, 'lambda': 1., 'nthread': -1, 'booster' : 'gbtree', 'silent': 1,
'eval_metric': 'rmse', 'objective': 'reg:linear'}
model = xgb.train(xgb_pars, dtrain, 10, watchlist, early_stopping_rounds=2,
      maximize=False, verbose_eval=1)
print('Modeling RMSLE %.5f' % model.best_score)

<h3>5.1 Hyperparameter Tuning and Feature Importance</h3>

<p>Model performance was further improved through hyperparameter tuning by adjusting parameters such as the learning rate, maximum tree depth, subsampling ratio, and feature sampling ratio. Care was taken to balance model complexity and prevent overfitting, ensuring good generalization on unseen data.</p>

<p>After training, XGBoost's feature importance scores were analyzed to identify the variables that contributed most to trip duration predictions. This analysis provides valuable insights into the relative impact of temporal, geographic, and route-based features on the model's performance.</p>

In [ ]:
xgb.plot_importance(model, max_num_features=28, height=0.7)

<h3>5.2 Feature Importance and Prediction</h3>

<p>Feature importance analysis revealed that distance-based features had the greatest influence on predicting trip duration, followed by temporal, geographic, and route-related features. These results align with expectations, as longer travel distances generally correspond to longer trip times.</p>

<p>After validating the model's performance, the trained XGBoost model was used to generate trip duration predictions for the test dataset, producing the final submission file for the Kaggle competition.</p>

In [ ]:
pred = model.predict(dtest)
pred = np.exp(pred) - 1

Time for Submission

In [ ]:
submission = pd.concat([Test_id, pd.DataFrame(pred)], axis=1)
submission.columns = ['id','trip_duration']
submission['trip_duration'] = submission.apply(lambda x : 1 if (x['trip_duration'] <= 0) else x['trip_duration'], axis = 1)
submission.to_csv("submission.csv", index=False)